# Raw 데이터 수집

국가법령정보 API에서 부동산/임대차 관련 문서를 수집하여 `data/raw/{target}.jsonl`에 저장한다.

| target  | 설명                  | 중복 제거 기준       |
| ------- | --------------------- | -------------------- |
| `acr`   | 국민권익위원회 결정문 | `결정문일련번호`     |
| `eflaw` | 현행법령              | `법령ID`             |
| `prec`  | 판례                  | `판례일련번호`       |
| `expc`  | 법령해석례            | `법령해석례일련번호` |

> 각 섹션은 독립적으로 실행 가능. 전체 실행 시 **Kernel → Restart & Run All**.


## 0. 공통 설정


In [1]:
import sys

sys.path.insert(0, "..")
from src.api import fetch_list, save_raw, fetch_details

## 2. 법령해석례(expc)


In [2]:
EXPC_QUERIES: list[str] = [
    "임대차",
    "보증금",
    "주택임대차",
    "전세",
    "임차인",
]

In [3]:
seen_expc: set[str] = set()  # 수집된 법정해석례 일련번호 추적
expc_items: list[dict] = []

for query in EXPC_QUERIES:
    items = fetch_list("expc", query=query, max_items=None, extra_params={"search": 2})
    for item in items:
        doc_id = str(item.get("법령해석례일련번호", ""))
        # doc_id가 비어있거나 이미 수집한 항목이면 skip
        if doc_id and doc_id not in seen_expc:
            seen_expc.add(doc_id)
            expc_items.append(item)

save_raw(expc_items, target="expc", mode="w")

print(len(expc_items))

365


In [4]:
expc_details = fetch_details(
  target="expc",
  items=expc_items,
  id_field="법령해석례일련번호",
)

# 목록만 있는 expc.jsonl 덮어쓰기
result = save_raw(expc_details, target="expc", mode="w")
print(len(expc_details))

365


## 3. 판례 (prec)


In [5]:
# 임대차 분쟁에서 자주 다뤄지는 판례 관련 키워드
PREC_QUERIES: list[str] = [
    "임대차",
    "보증금",
    "전세",
    "임차인",
    "계약갱신청구권",
    "원상회복",
    "수선의무",
    "명도",
]

In [6]:

seen_prec: set[str] = set() # 수집된 판례일련번호
prec_items: list[dict] = [] 

for query in PREC_QUERIES:
    items = fetch_list("prec", query=query, max_items=None, extra_params={"search": 2})
    for item in items:
        doc_id = str(item.get("판례일련번호", ""))
        # doc_id가 비어있거나 이미 수집한 항목이면 skip
        if doc_id and doc_id not in seen_prec:
            seen_prec.add(doc_id)
            prec_items.append(item)

save_raw(prec_items, target="prec", mode="w")

print(len(prec_items))

17502


In [ ]:
prec_details = fetch_details(
  target="prec",
  items=prec_items,
  id_field="판례일련번호",
)

# 목록만 있는 prec.jsonl 덮어쓰기
result = save_raw(prec_details, target="prec", mode="w")
print(len(prec_details))

## 4. 법령(eflaw)


In [ ]:
# 임대차 관련 법령 해석 키워드

EFLAW_QUERIES: list[str] = [
    "주택임대차보호법",
    "상가건물임대차보호법",
    "민법",
    "부동산",
    "전세사기",
]

In [ ]:
# eflaw (현행법령) 수집
seen_eflaw: set[str] = set() # 수집된 법령ID 추적
eflaw_items: list[dict] = []

for query in EFLAW_QUERIES:
    items = fetch_list("eflaw", query=query, max_items=None)
    for item in items:
        # doc_id가 비어있거나 이미 수집한 항목이면 skip
        doc_id = str(item.get("법령ID", ""))
        if doc_id and doc_id not in seen_eflaw:
            seen_eflaw.add(doc_id)
            eflaw_items.append(item)

save_raw(eflaw_items, target="eflaw", mode="w")
print(len(eflaw_items))

In [ ]:
# eflaw 본문 상세 수집
eflaw_details = fetch_details(
    target="eflaw",
    items=eflaw_items,
    id_field="법령ID",
    response_type="HTML",
)

result = save_raw(eflaw_details, target="eflaw", mode="w")
print(len(eflaw_details))

---

## 5. 수집 결과 요약


In [ ]:
from pathlib import Path

raw_dir = Path("../data/raw")

# 각 target의 .jsonl 파일 존재 여부 및 수집 건수 확인
# JSONL은 한 줄 = 한 건이므로 줄 수 = 수집 건수
for target in ["acr", "eflaw", "prec", "expc"]:
    path = raw_dir / f"{target}.jsonl"
    if path.exists():
        count = sum(1 for _ in path.open(encoding="utf-8"))
        print(f"{target:8s}: {count:5d}건  →  {path}")
    else:
        print(f"{target:8s}: 파일 없음")